# API call code for theoretical and operationalized SEPARATED prompt variants, on GPT-5. 
# We separate by 1. Title + Excerpt Frames 2. Binary tasks (demands present + quoted voices) 3. thematic present

In [28]:
import pandas as pd
df = pd.read_excel("frame_gold.xlsx")
df.head()

,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,authority_quote,protester_quote,thematic_present
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,no,yes,yes
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,yes,no,yes
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,yes,no,yes


# load prompts, df

In [31]:
# If needed:
# %pip install openai pandas openpyxl

import os
import json
import time
import pandas as pd

from openai import OpenAI

import separated_prompts
import importlib
importlib.reload(separated_prompts)

from frame_prompts_separated import (
    THEORETICAL_FRAMES_PROMPT,
    THEORETICAL_BINARY_PROMPT,
    THEORETICAL_THEMATIC_PROMPT,
)

# -----------------------------
# MODEL SETTINGS
# -----------------------------

MODEL = "gpt-5"
REASONING_EFFORT = "medium"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# -----------------------------
# LOAD GOLD DATA
# -----------------------------

df = pd.read_excel("frame_gold.xlsx")

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_columns].copy()

df["title"] = df["title"].fillna("").astype(str)
df["protest_excerpt"] = df["protest_excerpt"].fillna("").astype(str)


# Normalize gold labels
label_columns = [
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

for col in label_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

# In case your source file still uses "violence"
df["title_frame"] = df["title_frame"].replace(
    "violence", "protester_violence"
)

df["excerpt_frame"] = df["excerpt_frame"].replace(
    "violence", "protester_violence"
)


# -----------------------------
# TEST FIRST
# -----------------------------

test_df = df.head(3).copy()

# Change this ONE line when ready for all 120:
RUN_DF = df

# For full validation:
# RUN_DF = df

print(f"Running {len(RUN_DF)} samples.")

Running 120 samples.


# schemas and API helper

In [32]:
FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
    "residual",
]


# ============================================================
# STRUCTURED OUTPUT SCHEMAS
# ============================================================

FRAMES_SCHEMA = {
    "type": "object",
    "properties": {
        "title_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "excerpt_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
    },
    "required": [
        "title_frame",
        "excerpt_frame",
    ],
    "additionalProperties": False,
}


BINARY_SCHEMA = {
    "type": "object",
    "properties": {
        "demands_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "protester_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "authority_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
    },
    "required": [
        "demands_present",
        "protester_quote",
        "authority_quote",
    ],
    "additionalProperties": False,
}


THEMATIC_SCHEMA = {
    "type": "object",
    "properties": {
        "thematic_present": {
            "type": "string",
            "enum": ["yes", "no"],
        }
    },
    "required": ["thematic_present"],
    "additionalProperties": False,
}


# ============================================================
# FORMAT ARTICLE INPUT
# ============================================================

def format_input(title, excerpt):
    return f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""


# ============================================================
# GENERIC API CALL
# ============================================================

def call_classifier(
    title,
    excerpt,
    system_prompt,
    schema,
    schema_name,
    max_retries=3
):

    user_input = format_input(title, excerpt)

    for attempt in range(max_retries):

        try:
            response = client.responses.create(
                model=MODEL,
                instructions=system_prompt,
                input=user_input,

                reasoning={
                    "effort": REASONING_EFFORT
                },

                text={
                    "verbosity": "low",
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "strict": True,
                        "schema": schema,
                    },
                },
            )

            parsed = json.loads(response.output_text)

            # Token details
            cached_tokens = None
            reasoning_tokens = None

            try:
                cached_tokens = (
                    response.usage
                    .input_tokens_details
                    .cached_tokens
                )
            except Exception:
                pass

            try:
                reasoning_tokens = (
                    response.usage
                    .output_tokens_details
                    .reasoning_tokens
                )
            except Exception:
                pass

            return {
                **parsed,
                "response_id": response.id,
                "input_tokens": response.usage.input_tokens,
                "cached_tokens": cached_tokens,
                "output_tokens": response.usage.output_tokens,
                "reasoning_tokens": reasoning_tokens,
                "error": None,
            }

        except Exception as e:

            print(
                f"Attempt {attempt + 1}/{max_retries} failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

            else:
                return {
                    "response_id": None,
                    "input_tokens": None,
                    "cached_tokens": None,
                    "output_tokens": None,
                    "reasoning_tokens": None,
                    "error": str(e),
                }

# run all three tasks, save immediately, and merge by uuid into one final df

In [ ]:
# ============================================================
# 1. FRAMES
# ============================================================

frame_results = []

print("\n==============================")
print("RUNNING FRAME CLASSIFICATION")
print("==============================")

for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

    print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

    result = call_classifier(
        title=row["title"],
        excerpt=row["protest_excerpt"],
        system_prompt=THEORETICAL_FRAMES_PROMPT,
        schema=FRAMES_SCHEMA,
        schema_name="theoretical_frames",
    )

    result["uuid"] = row["uuid"]
    frame_results.append(result)


frames_df = pd.DataFrame(frame_results)

frames_df = frames_df.rename(columns={
    "title_frame": "gpt_title_frame",
    "excerpt_frame": "gpt_excerpt_frame",
    "response_id": "frames_response_id",
    "input_tokens": "frames_input_tokens",
    "cached_tokens": "frames_cached_tokens",
    "output_tokens": "frames_output_tokens",
    "reasoning_tokens": "frames_reasoning_tokens",
    "error": "frames_error",
})

frames_df.to_excel(
    "theoretical_separated_frames_v2.xlsx",
    index=False
)

print("Frames saved.")


# # ============================================================
# # 2. BINARY VARIABLES
# # ============================================================

# binary_results = []

# print("\n==============================")
# print("RUNNING BINARY CLASSIFICATION")
# print("==============================")

# for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

#     print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

#     result = call_classifier(
#         title=row["title"],
#         excerpt=row["protest_excerpt"],
#         system_prompt=THEORETICAL_BINARY_PROMPT,
#         schema=BINARY_SCHEMA,
#         schema_name="theoretical_binary",
#     )

#     result["uuid"] = row["uuid"]
#     binary_results.append(result)


# binary_df = pd.DataFrame(binary_results)

# binary_df = binary_df.rename(columns={
#     "demands_present": "gpt_demands_present",
#     "protester_quote": "gpt_protester_quote",
#     "authority_quote": "gpt_authority_quote",
#     "response_id": "binary_response_id",
#     "input_tokens": "binary_input_tokens",
#     "cached_tokens": "binary_cached_tokens",
#     "output_tokens": "binary_output_tokens",
#     "reasoning_tokens": "binary_reasoning_tokens",
#     "error": "binary_error",
# })

# binary_df.to_excel(
#     "theoretical_separated_binary_raw.xlsx",
#     index=False
# )

# print("Binary results saved.")


# # ============================================================
# # 3. THEMATIC
# # ============================================================

# thematic_results = []

# print("\n==============================")
# print("RUNNING THEMATIC CLASSIFICATION")
# print("==============================")

# for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

#     print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

#     result = call_classifier(
#         title=row["title"],
#         excerpt=row["protest_excerpt"],
#         system_prompt=THEORETICAL_THEMATIC_PROMPT,
#         schema=THEMATIC_SCHEMA,
#         schema_name="theoretical_thematic",
#     )

#     result["uuid"] = row["uuid"]
#     thematic_results.append(result)


# thematic_df = pd.DataFrame(thematic_results)

# thematic_df = thematic_df.rename(columns={
#     "thematic_present": "gpt_thematic_present",
#     "response_id": "thematic_response_id",
#     "input_tokens": "thematic_input_tokens",
#     "cached_tokens": "thematic_cached_tokens",
#     "output_tokens": "thematic_output_tokens",
#     "reasoning_tokens": "thematic_reasoning_tokens",
#     "error": "thematic_error",
# })

# thematic_df.to_excel(
#     "theoretical_separated_thematic_raw.xlsx",
#     index=False
# )

# print("Thematic results saved.")


# # ============================================================
# # 4. MERGE EVERYTHING BACK ON UUID
# # ============================================================

# results_df = (
#     RUN_DF
#     .merge(
#         frames_df,
#         on="uuid",
#         how="left",
#         validate="one_to_one"
#     )
#     .merge(
#         binary_df,
#         on="uuid",
#         how="left",
#         validate="one_to_one"
#     )
#     .merge(
#         thematic_df,
#         on="uuid",
#         how="left",
#         validate="one_to_one"
#     )
# )


# # ============================================================
# # 5. TOTAL TOKEN USAGE PER ARTICLE
# # ============================================================

# results_df["total_input_tokens"] = (
#     results_df["frames_input_tokens"].fillna(0)
#     + results_df["binary_input_tokens"].fillna(0)
#     + results_df["thematic_input_tokens"].fillna(0)
# )

# results_df["total_output_tokens"] = (
#     results_df["frames_output_tokens"].fillna(0)
#     + results_df["binary_output_tokens"].fillna(0)
#     + results_df["thematic_output_tokens"].fillna(0)
# )

# results_df["total_reasoning_tokens"] = (
#     results_df["frames_reasoning_tokens"].fillna(0)
#     + results_df["binary_reasoning_tokens"].fillna(0)
#     + results_df["thematic_reasoning_tokens"].fillna(0)
# )

# results_df["total_cached_tokens"] = (
#     results_df["frames_cached_tokens"].fillna(0)
#     + results_df["binary_cached_tokens"].fillna(0)
#     + results_df["thematic_cached_tokens"].fillna(0)
# )


# # ============================================================
# # 6. SAVE MERGED VALIDATION FILE
# # ============================================================

# output_file = (
#     f"merged_operationalized_separated_"
#     f"{MODEL}_{REASONING_EFFORT}.xlsx"
# )

# results_df.to_excel(
#     output_file,
#     index=False
# )

# print(f"\nMerged results saved to: {output_file}")


# # ============================================================
# # 7. QUICK TOKEN SUMMARY
# # ============================================================

# print("\nAverage tokens per ARTICLE across all 3 calls:")

# print(
#     f"Input:     "
#     f"{results_df['total_input_tokens'].mean():,.0f}"
# )

# print(
#     f"Cached:    "
#     f"{results_df['total_cached_tokens'].mean():,.0f}"
# )

# print(
#     f"Output:    "
#     f"{results_df['total_output_tokens'].mean():,.0f}"
# )

# print(
#     f"Reasoning: "
#     f"{results_df['total_reasoning_tokens'].mean():,.0f}"
# )

# results_df.head()

In [37]:
# Load frozen gold standard
gold_df = pd.read_excel("frame_gold.xlsx")

# Keep only the gold variables we need
gold_frames = gold_df[
    [
        "uuid",
        "title_frame",
        "excerpt_frame"
    ]
].copy()


# If necessary, normalize your violence label as before
gold_frames["title_frame"] = (
    gold_frames["title_frame"]
    .replace("violence", "protester_violence")
)

gold_frames["excerpt_frame"] = (
    gold_frames["excerpt_frame"]
    .replace("violence", "protester_violence")
)


# Merge gold onto saved GPT predictions
frames_eval_df = gold_frames.merge(
    frames_df,
    on="uuid",
    how="inner",
    validate="one_to_one"
)

print("Paired rows:", len(frames_eval_df))
print(frames_eval_df.columns.tolist())

frames_eval_df.to_excel("only_frames_v2.xlsx", index=False)

display(frames_eval_df.head())

Paired rows: 120
['uuid', 'title_frame', 'excerpt_frame', 'gpt_title_frame', 'gpt_excerpt_frame', 'frames_response_id', 'frames_input_tokens', 'frames_cached_tokens', 'frames_output_tokens', 'frames_reasoning_tokens', 'frames_error']


,uuid,title_frame,excerpt_frame,gpt_title_frame,gpt_excerpt_frame,frames_response_id,frames_input_tokens,frames_cached_tokens,frames_output_tokens,frames_reasoning_tokens,frames_error
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,residual,grievance,residual,grievance,resp_02626eaa6f92f115006a9c3b35ff0487d1a6b3211...,1339,0,514,448,NaN
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,residual,widespread_support,residual,grievance,resp_006700c755d77837006a9c3b3be36087d18c97a26...,1405,1024,495,448,NaN
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,grievance,disruption,disruption,disruption,resp_0a94e1331ba97089006a9c3b41ce6487d1b92bce5...,1397,1024,638,576,NaN
3,155faae5-c8b6-441e-a20e-dce06e01cf32,residual,grievance,residual,grievance,resp_0fbe7a6f5257da7a006a9c3b47b5c087d1bf5b649...,1341,1024,214,128,NaN
4,50fea669-6828-44de-8a70-b448e398be11,residual,protester_violence,residual,protester_violence,resp_0a4269b026bd7ea8006a9c3b4a957487d19eaea80...,1639,1024,1148,1088,NaN


In [39]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

import simpledorff


# ============================================================
# KRIPPENDORFF'S ALPHA
# ============================================================

def calculate_alpha(y_true, y_pred):

    alpha_df = pd.DataFrame({
        "item_id": np.arange(len(y_true)).repeat(2),
        "annotator_id": ["human", "gpt"] * len(y_true),
        "label": np.column_stack([
            y_true.to_numpy(),
            y_pred.to_numpy()
        ]).reshape(-1)
    })

    return simpledorff.calculate_krippendorffs_alpha_for_df(
        alpha_df,
        experiment_col="item_id",
        annotator_col="annotator_id",
        class_col="label"
    )


# ============================================================
# FRAME TASKS
# ============================================================

tasks = {
    "title_frame": {
        "gold": "title_frame",
        "pred": "gpt_title_frame",
    },

    "excerpt_frame": {
        "gold": "excerpt_frame",
        "pred": "gpt_excerpt_frame",
    },
}


# ============================================================
# CALCULATE METRICS
# ============================================================

summary_rows = []

for task_name, info in tasks.items():

    valid = frames_eval_df[
        [info["gold"], info["pred"]]
    ].dropna()

    y_true = valid[info["gold"]]
    y_pred = valid[info["pred"]]

    summary_rows.append({
        "task": task_name,
        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "kripp_alpha": calculate_alpha(
            y_true,
            y_pred
        ),
    })


# ============================================================
# SUMMARY DATAFRAME
# ============================================================

summary_df = pd.DataFrame(summary_rows)

metric_cols = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "kripp_alpha",
]

summary_df[metric_cols] = (
    summary_df[metric_cols]
    .round(3)
)

display(summary_df)

,task,n,accuracy,macro_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.758,0.600,0.776,0.597
1,excerpt_frame,120,0.675,0.615,0.668,0.612


# results summary table

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix
)
import simpledorff
import numpy as np

# --------------------------------------------------
# EVALUATION FUNCTIONS
# --------------------------------------------------



TASKS = {
    "title_frame": {
        "gold": "title_frame",
        "pred": "gpt_title_frame",
        "type": "multiclass",
    },
    "excerpt_frame": {
        "gold": "excerpt_frame",
        "pred": "gpt_excerpt_frame",
        "type": "multiclass",
    },
    "demands_present": {
        "gold": "demands_present",
        "pred": "gpt_demands_present",
        "type": "binary",
    },
    "protester_quote": {
        "gold": "protester_quote",
        "pred": "gpt_protester_quote",
        "type": "binary",
    },
    "authority_quote": {
        "gold": "authority_quote",
        "pred": "gpt_authority_quote",
        "type": "binary",
    },
    "thematic_present": {
        "gold": "thematic_present",
        "pred": "gpt_thematic_present",
        "type": "binary",
    },
}


def kripp_alpha_two_annotators(gold, pred):
    """
    Krippendorff's alpha for nominal labels,
    treating Human Gold and GPT as two annotators.
    """

    tmp = pd.DataFrame({
        "item_id": np.arange(len(gold)).repeat(2),
        "annotator_id": ["human", "gpt"] * len(gold),
        "label": np.column_stack([gold, pred]).reshape(-1),
    })

    return simpledorff.calculate_krippendorffs_alpha_for_df(
        tmp,
        experiment_col="item_id",
        annotator_col="annotator_id",
        class_col="label",
    )


def evaluate_task(data, gold_col, pred_col, task_type):

    valid = data[[gold_col, pred_col]].dropna()

    y_true = valid[gold_col].astype(str)
    y_pred = valid[pred_col].astype(str)

    result = {
        "n": len(valid),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "kripp_alpha": kripp_alpha_two_annotators(
            y_true.tolist(),
            y_pred.tolist(),
        ),
    }

    if task_type == "binary":

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=["yes"],
            average=None,
            zero_division=0,
        )

        result["yes_precision"] = precision[0]
        result["yes_recall"] = recall[0]
        result["yes_f1"] = f1[0]

    else:

        result["weighted_f1"] = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )

    return result


# --------------------------------------------------
# SUMMARY TABLE
# --------------------------------------------------

summary_rows = []

for task_name, config in TASKS.items():

    metrics = evaluate_task(
        results_df,
        gold_col=config["gold"],
        pred_col=config["pred"],
        task_type=config["type"],
    )

    summary_rows.append({
        "task": task_name,
        **metrics,
    })


summary_df = pd.DataFrame(summary_rows)

# Arrange the most useful columns first.
preferred_cols = [
    "task",
    "n",
    "accuracy",
    "macro_f1",
    "yes_precision",
    "yes_recall",
    "yes_f1",
    "weighted_f1",
    "kripp_alpha",
]

summary_df = summary_df[
    [c for c in preferred_cols if c in summary_df.columns]
]

print("\n==============================")
print("VALIDATION SUMMARY")
print("==============================\n")

display(
    summary_df.round(3)
)


# --------------------------------------------------
# DETAILED MULTICLASS REPORTS
# --------------------------------------------------

for task in ["title_frame", "excerpt_frame"]:

    gold_col = TASKS[task]["gold"]
    pred_col = TASKS[task]["pred"]

    valid = results_df[[gold_col, pred_col]].dropna()

    print("\n")
    print("=" * 70)
    print(task.upper())
    print("=" * 70)

    print(
        classification_report(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
            zero_division=0,
        )
    )

    cm = pd.DataFrame(
        confusion_matrix(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
        ),
        index=[f"gold_{x}" for x in FRAME_LABELS],
        columns=[f"gpt_{x}" for x in FRAME_LABELS],
    )

    display(cm)


# --------------------------------------------------
# SAVE summary
# --------------------------------------------------
summary_df.to_excel(
    f"summary_operationalized_separated_gpt-5-mini.xlsx",
    index=False
)


VALIDATION SUMMARY



,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.792,0.649,NaN,NaN,NaN,0.802,0.647
1,excerpt_frame,120,0.642,0.580,NaN,NaN,NaN,0.630,0.572
2,demands_present,120,0.842,0.800,0.848,0.940,0.891,NaN,0.601
3,protester_quote,120,0.858,0.855,0.827,0.843,0.835,NaN,0.712
4,authority_quote,120,0.767,0.713,0.742,0.960,0.837,NaN,0.428
5,thematic_present,120,0.783,0.782,0.724,0.808,0.764,NaN,0.565




TITLE_FRAME
                    precision    recall  f1-score   support

protester_violence       0.50      0.60      0.55         5
harm_to_protesters       0.62      0.83      0.71         6
   property_damage       0.50      1.00      0.67         1
        disruption       0.29      0.50      0.36         4
         grievance       0.75      0.86      0.80        14
widespread_support       0.83      0.83      0.83         6
          peaceful       0.00      0.00      0.00         0
         impactful       0.40      0.40      0.40         5
          residual       0.93      0.82      0.87        79

          accuracy                           0.79       120
         macro avg       0.54      0.65      0.58       120
      weighted avg       0.82      0.79      0.80       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,3,0,1,0,0,0,0,0,1
gold_harm_to_protesters,0,5,0,0,0,0,0,0,1
gold_property_damage,0,0,1,0,0,0,0,0,0
gold_disruption,0,0,0,2,0,0,0,1,1
gold_grievance,0,0,0,1,12,0,0,0,1
gold_widespread_support,0,0,0,0,1,5,0,0,0
gold_peaceful,0,0,0,0,0,0,0,0,0
gold_impactful,0,0,0,2,0,0,0,2,1
gold_residual,3,3,0,2,3,1,0,2,65




EXCERPT_FRAME
                    precision    recall  f1-score   support

protester_violence       0.77      0.74      0.75        31
harm_to_protesters       0.69      0.92      0.79        12
   property_damage       0.75      0.75      0.75         4
        disruption       0.64      0.41      0.50        17
         grievance       0.62      0.80      0.70        25
widespread_support       0.67      0.29      0.40        14
          peaceful       0.25      0.33      0.29         3
         impactful       0.42      0.62      0.50         8
          residual       0.60      0.50      0.55         6

          accuracy                           0.64       120
         macro avg       0.60      0.60      0.58       120
      weighted avg       0.65      0.64      0.63       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,23,2,0,0,1,0,2,2,1
gold_harm_to_protesters,1,11,0,0,0,0,0,0,0
gold_property_damage,1,0,3,0,0,0,0,0,0
gold_disruption,5,0,0,7,2,1,0,2,0
gold_grievance,0,1,0,1,20,1,1,1,0
gold_widespread_support,0,1,1,1,7,4,0,0,0
gold_peaceful,0,0,0,0,1,0,1,1,0
gold_impactful,0,0,0,2,0,0,0,5,1
gold_residual,0,1,0,0,1,0,0,1,3


In [15]:
import pandas as pd

summary_old = pd.read_excel(
    "summary_theoretical_separated.xlsx"
)

display(summary_old)

,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.758333,0.593236,NaN,NaN,NaN,0.769434,0.587440
1,excerpt_frame,120,0.683333,0.617820,NaN,NaN,NaN,0.674457,0.621126
2,demands_present,120,0.833333,0.791014,0.846154,0.927711,0.885057,NaN,0.583769
3,protester_quote,120,0.850000,0.847285,0.811321,0.843137,0.826923,NaN,0.695843
4,authority_quote,120,0.750000,0.686520,0.727273,0.960000,0.827586,NaN,0.375653
5,thematic_present,120,0.775000,0.773727,0.711864,0.807692,0.756757,NaN,0.549340
